# Document Ingestion Testing

In [1]:
# packages

## link project directory
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

## custom packages
from ingestion.documents.sec import SecEdgarProvider

## data collection
import sqlite3
import pandas as pd

In [2]:
# constants
from constants import DEFAULT_SQLITE_PATH

In [12]:
# connect to data
connection = sqlite3.connect(DEFAULT_SQLITE_PATH)
connection.row_factory = sqlite3.Row

# retrieve CIKs
query = "select distinct company_id, cik from companies"
ciks = pd.read_sql_query(query, connection).set_index('company_id').to_dict()['cik']
ciks


{'AAPL': '0000320193',
 'AMZN': '0001018724',
 'AVGO': '0001730168',
 'BRK-B': '0001067983',
 'GOOG': '0001652044',
 'GOOGL': '0001652044',
 'META': '0001326801',
 'MSFT': '0000789019',
 'NVDA': '0001045810',
 'TSLA': '0001318605'}

## Test SEC Retrieval

In [4]:
# # define the provide and get filings
# provider = SecEdgarProvider(
#     cik = ciks['cik'].iloc[1],
#     user_agent = "ai-investment-decision-support (nccruickshank94@gmail.com)"
# )

# filings = provider.list_filings(
#     filing_types = ['10-K', '10-Q'],
#     limit = 5
# )

# filings[0]

__TODO:__ delete `sec_v1.py` once you're confident you have what you need

In [ ]:
# test it


filings = provider.list_filings(
    cik = ciks['GOOG'],
    filing_types = ['10-K', '10-Q'],
    limit = 500
)

# test = provider.download_filing(filings)
filings[0].accession_number # how to save the file
filings[0].filing_url

'https://www.sec.gov/Archives/edgar/data/1652044/000165204426000071/goog-20260630.htm'

In [ ]:
# iterate over the CIKs and get all filings
from tqdm import tqdm

USER_AGENT = "ai-investment-decision-support (nccruickshank94@gmail.com)"
FILING_TYPES = ['10-K', '10-Q']
LIMIT = 500 # arbitrary high limit

# define the provider
provider = SecEdgarProvider(user_agent =  USER_AGENT)

# iterate through each company to get the filings
# ciks = {'GOOG': <cik>, 'AAPL': <cik>, ...}
for company, cik in ciks.items():
    # get the filings
    filings = provider.list_filings(
        cik = cik,
        filing_types = FILING_TYPES,
        limit = LIMIT
    )

    # store each of the retrieved HTML documents
    for f in tqdm(filings, desc = f'[{company}] Downloading filings', unit = 'filing'):
        # download the filing
        html_doc = provider.download_filing(f)
        
        # store the document in the database
        fpath = PROJECT_ROOT / 'data' / 'raw' / 'sec_edgar' / company / f'{f.accession_number}.html'
        with open(fpath, 'w') as f:
            f.write(html_doc)

Getting filings from AAPL (CIK = 0000320193)


[AAPL] Downloading filings: 100%|██████████| 45/45 [00:02<00:00, 20.98filing/s]


Getting filings from AMZN (CIK = 0001018724)


[AMZN] Downloading filings: 100%|██████████| 25/25 [00:01<00:00, 18.04filing/s]


Getting filings from AVGO (CIK = 0001730168)


...

[MSFT] Downloading filings: 100%|██████████| 26/26 [00:01<00:00, 21.02filing/s]


Getting filings from NVDA (CIK = 0001045810)


[NVDA] Downloading filings: 100%|██████████| 24/24 [00:01<00:00, 19.96filing/s]


Getting filings from TSLA (CIK = 0001318605)


[TSLA] Downloading filings: 100%|██████████| 34/34 [00:01<00:00, 24.60filing/s]
